In [1]:
#Intention Detection based on the Subcat method

In [6]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Tuple, Set
from collections import defaultdict

import pandas as pd

# ---- Optional stemming ----
try:
    from nltk.stem.snowball import SnowballStemmer
    _stemmer = SnowballStemmer("english")

    def _stem(token: str) -> str:
        return _stemmer.stem(token)
except Exception:
    def _stem(token: str) -> str:
        return token


# =========================
# I/O (edit these)
# =========================
BASE_DIR = Path(r"D:\3 - RQ3_2\Intention_3")  # <-- change as needed

INPUT_CSV = BASE_DIR / "All_episodes_with_messages.csv"

# IMPORTANT: point this to the FIXED dictionary
DICTIONARY_CSV = BASE_DIR / "dictionary.csv"

OUTPUT_CSV = BASE_DIR / "All_episodes_with_messages_with_intentions_subcat.csv"


# =========================
# Boost coefficients (as requested)
# =========================
ISSUE_BOOST = 3.0
COMMIT_BOOST = 1.0
PR_BOOST = 1.0


# =========================
# Episode schema columns
# =========================
END_COMMIT_COL = "episode_end_commit_sha"

# Start boundary fields
START_PARTS: List[Tuple[List[str], float]] = [
    (["start_commit_subject", "start_commit_body"], COMMIT_BOOST),
    (["start_pr_title", "start_pr_body"], PR_BOOST),

    # Boost ALL three issue columns equally
    (["start_issue_title"], ISSUE_BOOST),
    (["start_issue_body"], ISSUE_BOOST),
    (["start_issue_comments"], ISSUE_BOOST),

    # Optional: keep summary (not part of the "three issue columns")
    (["start_issue_summary"], 1.1),
]

# End boundary fields
END_PARTS: List[Tuple[List[str], float]] = [
    (["end_commit_subject", "end_commit_body"], COMMIT_BOOST),
    (["end_pr_title", "end_pr_body"], PR_BOOST),

    # Boost ALL three issue columns equally
    (["end_issue_title"], ISSUE_BOOST),
    (["end_issue_body"], ISSUE_BOOST),
    (["end_issue_comments"], ISSUE_BOOST),

    (["end_issue_summary"], 1.1),
]


# =========================
# Scoring / selection params
# =========================
MIN_SCORE = 1.5          # raise to improve precision; lower to improve recall
MULTI_RATIO = 0.80       # include labels within 80% of top score
MAX_LABELS = 3

# confidence_value = 0.6*share_selected + 0.4*margin_selected
CONF_W_SHARE = 0.6
CONF_W_MARGIN = 0.4

# confidence buckets
HIGH_TH = 0.75
MED_TH = 0.55


# =========================
# Tokenization helpers
# =========================
def _safe_str(v) -> str:
    if v is None:
        return ""
    if isinstance(v, float) and pd.isna(v):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s


def normalize_text_for_tokens(text: str) -> str:
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)  # de-camelcase
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


def tokenize_and_stem(text: str) -> List[str]:
    """
    Keeps digits (e2e, v2, aab, etc). Stems only purely alphabetic tokens.
    """
    if not text:
        return []
    text = normalize_text_for_tokens(text)
    raw = re.findall(r"[a-z0-9]+", text)
    out: List[str] = []
    for t in raw:
        if any(ch.isdigit() for ch in t):
            out.append(t)
        else:
            out.append(_stem(t))
    return out


def count_phrase_occurrences(tokens: List[str], phrase_tokens: List[str]) -> int:
    if not phrase_tokens or not tokens:
        return 0
    if len(phrase_tokens) == 1:
        p = phrase_tokens[0]
        return sum(1 for t in tokens if t == p)
    n = len(phrase_tokens)
    cnt = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            cnt += 1
    return cnt


# =========================
# Dictionary loading (LONG format)
# =========================
def load_dictionary_long(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Expects columns: label, keyword, optional weight, optional group.
    If label == "Blacklist" (case-insensitive), goes to blacklist.
    """
    df = pd.read_csv(path)

    colmap = {str(c).strip().lower(): c for c in df.columns}
    if "label" not in colmap or "keyword" not in colmap:
        raise ValueError("dictionary.csv must have columns: label, keyword")

    label_col = colmap["label"]
    keyword_col = colmap["keyword"]
    weight_col = colmap.get("weight", None)
    group_col = colmap.get("group", None)

    dict_terms: Dict[str, List[Dict]] = defaultdict(list)
    blacklist_terms: List[Dict] = []

    for _, row in df.iterrows():
        lab = _safe_str(row.get(label_col, "")).strip()
        kw = _safe_str(row.get(keyword_col, "")).strip()
        if not lab or not kw:
            continue

        wt = 1.0
        if weight_col is not None:
            try:
                wt = float(row.get(weight_col, 1.0))
            except Exception:
                wt = 1.0

        grp = _safe_str(row.get(group_col, "")).strip().upper() if group_col is not None else ""

        stem_tokens = tokenize_and_stem(kw)
        if not stem_tokens:
            continue

        item = {"keyword": kw, "weight": wt, "stem_tokens": stem_tokens, "group": grp}

        if lab.lower() == "blacklist":
            blacklist_terms.append(item)
        else:
            dict_terms[lab].append(item)

    # de-dup by stem sequence within label
    def dedup(items: List[Dict]) -> List[Dict]:
        seen = set()
        out = []
        for it in items:
            k = " ".join(it["stem_tokens"])
            if k in seen:
                continue
            seen.add(k)
            out.append(it)
        return out

    dict_terms = {lab: dedup(items) for lab, items in dict_terms.items()}
    blacklist_terms = dedup(blacklist_terms)

    if not dict_terms:
        raise ValueError("No label keywords loaded from dictionary.csv")

    return dict_terms, blacklist_terms


# =========================
# Build boundary parts
# =========================
def build_parts_from_row(row: pd.Series, spec: List[Tuple[List[str], float]]) -> List[Tuple[str, float]]:
    """
    spec: list of ([columns], multiplier)
    returns: list of (text, multiplier)
    """
    parts: List[Tuple[str, float]] = []
    for cols, mult in spec:
        txt = "\n".join([_safe_str(row.get(c, "")) for c in cols]).strip()
        if txt:
            parts.append((txt, mult))
    return parts


# =========================
# Constraints using GLOBAL groups (important)
# =========================
def passes_constraints(label: str, global_groups: Set[str]) -> bool:
    """
    Constraints based on ANY matched groups in the boundary (global).
    """
    if label == "Introduce / strengthen CI-backed tests":
        return ("CI" in global_groups) and ("TEST" in global_groups)

    if label == "Migrate or modernise CI infrastructure":
        return ("MIGRATE" in global_groups) and ("CI" in global_groups)

    if label == "Clean up or simplify CI / environment configuration":
        return ("CLEANUP" in global_groups) and (("CI" in global_groups) or ("ENV" in global_groups))

    # NEW: PERF/Stability is only valid when it looks like CI/TEST stability,
    # not app-only perf. We treat PERF or FAIL as the stability signal.
    if label == "Address performance or stability issues":
        return (("PERF" in global_groups) or ("FAIL" in global_groups)) and (("CI" in global_groups) or ("TEST" in global_groups))

    return True


# =========================
# Scoring
# =========================
def classify_parts(
    parts: List[Tuple[str, float]],
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    cap_per_keyword_per_part: bool = True,
) -> Tuple[Dict[str, float], Dict[str, List[str]], List[str], Set[str]]:
    """
    Returns:
      scores[label] = float
      matched_keywords[label] = [kw...]
      blacklist_hits = [kw...]
      global_groups = {GROUP...} across all matches
    """
    token_parts: List[Tuple[List[str], float]] = []
    for text, mult in parts:
        toks = tokenize_and_stem(text)
        if toks:
            token_parts.append((toks, mult))

    scores = {label: 0.0 for label in dict_terms}
    matched_keywords: Dict[str, List[str]] = {label: [] for label in dict_terms}

    # blacklist hits
    blacklist_hits: List[str] = []
    for it in blacklist_terms:
        for toks, _mult in token_parts:
            if count_phrase_occurrences(toks, it["stem_tokens"]) > 0:
                blacklist_hits.append(it["keyword"])
                break
    blacklist_hits = sorted(set(blacklist_hits))

    global_groups: Set[str] = set()

    # label scoring
    for label, items in dict_terms.items():
        for it in items:
            total_occ = 0.0
            hit = False
            for toks, mult in token_parts:
                occ = count_phrase_occurrences(toks, it["stem_tokens"])
                if occ > 0:
                    hit = True
                    if cap_per_keyword_per_part:
                        occ = 1
                    total_occ += (occ * mult)

            if hit:
                scores[label] += it["weight"] * total_occ
                matched_keywords[label].append(it["keyword"])

                g = (it.get("group") or "").strip().upper()
                if g:
                    global_groups.add(g)

    # Apply constraints based on global groups
    for label in list(scores.keys()):
        if not passes_constraints(label, global_groups):
            scores[label] = 0.0
            matched_keywords[label] = []

    return scores, matched_keywords, blacklist_hits, global_groups


# =========================
# Precedence rule: PERF beats CI-backed-tests when CI/TEST + stability signal exists
# =========================
def apply_precedence(scores: Dict[str, float], matched_keywords: Dict[str, List[str]], global_groups: Set[str]) -> None:
    """
    If stability evidence (PERF or FAIL) appears alongside CI/TEST,
    prefer PERF/Stability over CI-backed-tests.
    """
    perf_lab = "Address performance or stability issues"
    ci_lab = "Introduce / strengthen CI-backed tests"

    has_stability_signal = ("PERF" in global_groups) or ("FAIL" in global_groups)
    has_ci_or_test = ("CI" in global_groups) or ("TEST" in global_groups)

    if has_stability_signal and has_ci_or_test:
        # Suppress CI-backed-tests so it cannot win in these cases
        scores[ci_lab] = 0.0
        matched_keywords[ci_lab] = []

        # Nudge PERF upward so it wins ties
        if scores.get(perf_lab, 0.0) > 0:
            scores[perf_lab] *= 1.5


# =========================
# Selection + confidence
# =========================
def assign_labels_multi(
    scores: Dict[str, float],
    matched_keywords: Dict[str, List[str]],
    blacklist_hits: List[str],
    min_score: float = MIN_SCORE,
    multi_ratio: float = MULTI_RATIO,
    max_labels: int = MAX_LABELS,
) -> Dict[str, object]:
    """
    Select labels based on score proximity to top score.
    """
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0] if items else ("", 0.0)
    second_label, second_score = items[1] if len(items) > 1 else ("", 0.0)

    total = float(sum(scores.values()))
    if top_score < min_score:
        return {
            "label_str": None,
            "top_label": top_label, "top_score": float(top_score),
            "second_label": second_label, "second_score": float(second_score),
            "total_score": float(total),
            "selected_score_sum": 0.0,
            "confidence_share_selected": 0.0,
            "confidence_margin_selected": 0.0,
            "confidence_value": 0.0,
            "confidence_level": "UNLABELED_NOISE" if blacklist_hits else "UNLABELED",
            "matched_keywords": "",
            "blacklist_hits": "; ".join(blacklist_hits),
        }

    chosen: List[str] = []
    chosen_scores: List[float] = []

    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
            chosen_scores.append(sc)
        if len(chosen) >= max_labels:
            break

    chosen_set = set(chosen)
    next_unselected = 0.0
    for lab, sc in items:
        if lab not in chosen_set:
            next_unselected = sc
            break

    selected_sum = float(sum(chosen_scores))
    share_selected = (selected_sum / total) if total > 0 else 0.0
    min_selected = float(min(chosen_scores)) if chosen_scores else 0.0
    margin_selected = ((min_selected - next_unselected) / min_selected) if min_selected > 0 else 0.0
    margin_selected = max(0.0, min(1.0, margin_selected))

    confidence_value = (CONF_W_SHARE * share_selected) + (CONF_W_MARGIN * margin_selected)

    if confidence_value >= HIGH_TH and selected_sum >= 2.5:
        level = "HIGH"
    elif confidence_value >= MED_TH:
        level = "MEDIUM"
    else:
        level = "LOW"

    matched = sorted({kw for lab in chosen for kw in matched_keywords.get(lab, [])})

    return {
        "label_str": " || ".join(chosen),
        "top_label": top_label, "top_score": float(top_score),
        "second_label": second_label, "second_score": float(second_score),
        "total_score": float(total),
        "selected_score_sum": float(selected_sum),
        "confidence_share_selected": float(share_selected),
        "confidence_margin_selected": float(margin_selected),
        "confidence_value": float(confidence_value),
        "confidence_level": level,
        "matched_keywords": "; ".join(matched),
        "blacklist_hits": "; ".join(blacklist_hits),
    }


def label_boundary(row: pd.Series, parts_spec, dict_terms, blacklist_terms) -> Dict[str, object]:
    parts = build_parts_from_row(row, parts_spec)
    scores, matched_keywords, blacklist_hits, global_groups = classify_parts(parts, dict_terms, blacklist_terms)

    # Apply precedence: stability wins over CI-backed-tests when both appear
    apply_precedence(scores, matched_keywords, global_groups)

    return assign_labels_multi(scores, matched_keywords, blacklist_hits)


# =========================
# Main
# =========================
def main() -> None:
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
    if not DICTIONARY_CSV.exists():
        raise FileNotFoundError(f"Dictionary CSV not found: {DICTIONARY_CSV}")

    df = pd.read_csv(INPUT_CSV)
    dict_terms, blacklist_terms = load_dictionary_long(DICTIONARY_CSV)

    start_results = []
    end_results = []

    for _, row in df.iterrows():
        # start: always label
        s = label_boundary(row, START_PARTS, dict_terms, blacklist_terms)
        start_results.append(s)

        # end: only if end commit exists
        if pd.isna(row.get(END_COMMIT_COL)):
            end_results.append({
                "label_str": None,
                "top_label": "", "top_score": 0.0,
                "second_label": "", "second_score": 0.0,
                "total_score": 0.0,
                "selected_score_sum": 0.0,
                "confidence_share_selected": 0.0,
                "confidence_margin_selected": 0.0,
                "confidence_value": 0.0,
                "confidence_level": "NO_END_COMMIT",
                "matched_keywords": "",
                "blacklist_hits": "",
            })
        else:
            e = label_boundary(row, END_PARTS, dict_terms, blacklist_terms)
            end_results.append(e)

    s_df = pd.DataFrame(start_results).add_prefix("start_")
    e_df = pd.DataFrame(end_results).add_prefix("end_")

    out = pd.concat([df, s_df, e_df], axis=1)
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print("[ok] wrote:", OUTPUT_CSV)


if __name__ == "__main__":
    main()


[ok] wrote: D:\3 - RQ3_2\Intention_3\All_episodes_with_messages_with_intentions_subcat.csv
